<a href="https://colab.research.google.com/github/kartik815/Amazon-ML-Challenge-2026/blob/main/notebooks/03_Normalization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [1]:
import os
import re
import gc
import pandas as pd


# ------------------------------------------------------------
# BASE PATH
# ------------------------------------------------------------

DRIVE_ROOT = "/content/drive/MyDrive/Amazon ML Challenge 2026"


# ------------------------------------------------------------
# DATASET PATH
# ------------------------------------------------------------

DATASET_ROOT = os.path.join(
    DRIVE_ROOT,
    "01_Dataset",
    "6ab10eb3b23ba_student_resource",
    "student_resource",
    "dataset"
)


TRAIN_ROOT = os.path.join(
    DATASET_ROOT,
    "train"
)

TEST_ROOT = os.path.join(
    DATASET_ROOT,
    "test"
)


EXPERIMENTS_ROOT = os.path.join(
    DRIVE_ROOT,
    "03_Experiments"
)

CLEANED_DATA_ROOT = os.path.join(
    EXPERIMENTS_ROOT,
    "Cleaned_Data"
)


# Create output directory if it doesn't exist
os.makedirs(
    CLEANED_DATA_ROOT,
    exist_ok=True
)


print("Dataset root:")
print(DATASET_ROOT)

print("\nTrain root:")
print(TRAIN_ROOT)

print("\nTest root:")
print(TEST_ROOT)

print("\nCleaned data output:")
print(CLEANED_DATA_ROOT)

Dataset root:
/content/drive/MyDrive/Amazon ML Challenge 2026/01_Dataset/6ab10eb3b23ba_student_resource/student_resource/dataset

Train root:
/content/drive/MyDrive/Amazon ML Challenge 2026/01_Dataset/6ab10eb3b23ba_student_resource/student_resource/dataset/train

Test root:
/content/drive/MyDrive/Amazon ML Challenge 2026/01_Dataset/6ab10eb3b23ba_student_resource/student_resource/dataset/test

Cleaned data output:
/content/drive/MyDrive/Amazon ML Challenge 2026/03_Experiments/Cleaned_Data


In [2]:
def load_source(path):

    return pd.read_csv(
        path,
        sep="\t",
        dtype={
            "entity_id": "string",
            "business_name": "string",
            "business_address": "string",
            "country": "string"
        }
    )

In [3]:
def normalize_basic_series(series):

    return (
        series
        .fillna("")
        .astype("string")

        # Unicode normalization
        .str.normalize("NFKC")

        # Lowercase
        .str.lower()

        # Common separators → spaces
        .str.replace(
            r"[/\\|,_;:]+",
            " ",
            regex=True
        )

        # Other punctuation → spaces
        .str.replace(
            r"[^\w\s]",
            " ",
            regex=True
        )

        # Collapse whitespace
        .str.replace(
            r"\s+",
            " ",
            regex=True
        )

        .str.strip()
    )

In [4]:
LEGAL_SUFFIX_MAP = {
    "limited": "ltd",
    "ltd": "ltd",

    "incorporated": "inc",
    "inc": "inc",

    "corporation": "corp",
    "corp": "corp",

    "private": "pvt",
    "pvt": "pvt",

    "company": "co",
    "co": "co",

    "llc": "llc",
    "llp": "llp",
    "plc": "plc"
}


def normalize_name_series(series):

    result = normalize_basic_series(series)

    for old, new in LEGAL_SUFFIX_MAP.items():

        result = result.str.replace(
            rf"\b{re.escape(old)}\b",
            new,
            regex=True
        )

    return result

In [5]:

ADDRESS_ABBREVIATIONS = {
    "street": "st",
    "st": "st",

    "road": "rd",
    "rd": "rd",

    "avenue": "ave",
    "ave": "ave",

    "boulevard": "blvd",
    "blvd": "blvd",

    "drive": "dr",
    "dr": "dr",

    "lane": "ln",
    "ln": "ln",

    "parkway": "pkwy",
    "pkwy": "pkwy",

    "highway": "hwy",
    "hwy": "hwy",

    "place": "pl",
    "pl": "pl",

    "court": "ct",
    "ct": "ct",

    "square": "sq",
    "sq": "sq",

    "apartment": "apt",
    "apt": "apt",

    "suite": "ste",
    "ste": "ste",

    "building": "bldg",
    "bldg": "bldg"
}


def normalize_address_series(series):

    result = normalize_basic_series(series)

    for old, new in ADDRESS_ABBREVIATIONS.items():

        result = result.str.replace(
            rf"\b{re.escape(old)}\b",
            new,
            regex=True
        )

    return result

In [6]:
def normalize_country_series(series):

    return (
        series
        .fillna("")
        .astype("string")
        .str.strip()
        .str.lower()
        .str.replace(
            r"\s+",
            " ",
            regex=True
        )
    )

In [7]:
def add_cleaned_columns(df):

    df["name_norm"] = normalize_name_series(
        df["business_name"]
    )

    df["address_norm"] = normalize_address_series(
        df["business_address"]
    )

    df["country_norm"] = normalize_country_series(
        df["country"]
    )


    df["name_missing"] = (
        df["name_norm"].str.len() == 0
    ).astype("int8")

    df["address_missing"] = (
        df["address_norm"].str.len() == 0
    ).astype("int8")



    df["name_token_count"] = (
        df["name_norm"]
        .str.count(r"\S+")
        .astype("int16")
    )

    df["address_token_count"] = (
        df["address_norm"]
        .str.count(r"\S+")
        .astype("int16")
    )

    return df

In [8]:
FILES = [

    (
        "train_s1",
        os.path.join(
            TRAIN_ROOT,
            "train_source1.tsv"
        )
    ),

    (
        "train_s2",
        os.path.join(
            TRAIN_ROOT,
            "train_source2.tsv"
        )
    ),

    (
        "train_s3",
        os.path.join(
            TRAIN_ROOT,
            "train_source3.tsv"
        )
    ),

    (
        "test_s1",
        os.path.join(
            TEST_ROOT,
            "test_source1.tsv"
        )
    ),

    (
        "test_s2",
        os.path.join(
            TEST_ROOT,
            "test_source2.tsv"
        )
    ),

    (
        "test_s3",
        os.path.join(
            TEST_ROOT,
            "test_source3.tsv"
        )
    )
]


print("Datasets to process:")

for name, path in FILES:
    print(f"{name}: {path}")

Datasets to process:
train_s1: /content/drive/MyDrive/Amazon ML Challenge 2026/01_Dataset/6ab10eb3b23ba_student_resource/student_resource/dataset/train/train_source1.tsv
train_s2: /content/drive/MyDrive/Amazon ML Challenge 2026/01_Dataset/6ab10eb3b23ba_student_resource/student_resource/dataset/train/train_source2.tsv
train_s3: /content/drive/MyDrive/Amazon ML Challenge 2026/01_Dataset/6ab10eb3b23ba_student_resource/student_resource/dataset/train/train_source3.tsv
test_s1: /content/drive/MyDrive/Amazon ML Challenge 2026/01_Dataset/6ab10eb3b23ba_student_resource/student_resource/dataset/test/test_source1.tsv
test_s2: /content/drive/MyDrive/Amazon ML Challenge 2026/01_Dataset/6ab10eb3b23ba_student_resource/student_resource/dataset/test/test_source2.tsv
test_s3: /content/drive/MyDrive/Amazon ML Challenge 2026/01_Dataset/6ab10eb3b23ba_student_resource/student_resource/dataset/test/test_source3.tsv


In [9]:
for name, input_path in FILES:

    print("\n" + "=" * 70)
    print("PROCESSING:", name)
    print("=" * 70)


    df = load_source(input_path)

    original_row_count = len(df)

    print(
        "Original rows:",
        original_row_count
    )

    df = add_cleaned_columns(df)

    assert "entity_id" in df.columns

    print("entity_id preserved: YES")

    assert len(df) == original_row_count

    print("Row count preserved:", len(df))

    output_path = os.path.join(
        CLEANED_DATA_ROOT,
        f"{name}_cleaned.tsv"
    )

    df.to_csv(
        output_path,
        sep="\t",
        index=False
    )

    print("\nSaved cleaned TSV:")
    print(output_path)

    del df

    gc.collect()

    print("Dataset released from RAM.")


print("\n" + "=" * 70)
print("ALL DATASETS CLEANED AND EXPORTED")
print("=" * 70)


PROCESSING: train_s1
Original rows: 2206821
entity_id preserved: YES
Row count preserved: 2206821

Saved cleaned TSV:
/content/drive/MyDrive/Amazon ML Challenge 2026/03_Experiments/Cleaned_Data/train_s1_cleaned.tsv
Dataset released from RAM.

PROCESSING: train_s2
Original rows: 5034616
entity_id preserved: YES
Row count preserved: 5034616

Saved cleaned TSV:
/content/drive/MyDrive/Amazon ML Challenge 2026/03_Experiments/Cleaned_Data/train_s2_cleaned.tsv
Dataset released from RAM.

PROCESSING: train_s3
Original rows: 5285603
entity_id preserved: YES
Row count preserved: 5285603

Saved cleaned TSV:
/content/drive/MyDrive/Amazon ML Challenge 2026/03_Experiments/Cleaned_Data/train_s3_cleaned.tsv
Dataset released from RAM.

PROCESSING: test_s1
Original rows: 1732544
entity_id preserved: YES
Row count preserved: 1732544

Saved cleaned TSV:
/content/drive/MyDrive/Amazon ML Challenge 2026/03_Experiments/Cleaned_Data/test_s1_cleaned.tsv
Dataset released from RAM.

PROCESSING: test_s2
Original 

In [10]:
EXPECTED_NEW_COLUMNS = [
    "name_norm",
    "address_norm",
    "country_norm",
    "name_missing",
    "address_missing",
    "name_token_count",
    "address_token_count"
]


for name, _ in FILES:

    cleaned_path = os.path.join(
        CLEANED_DATA_ROOT,
        f"{name}_cleaned.tsv"
    )

    print("\n" + "-" * 70)
    print("VERIFYING:", name)

    df = pd.read_csv(
        cleaned_path,
        sep="\t",
        dtype={
            "entity_id": "string",
            "business_name": "string",
            "business_address": "string",
            "country": "string"
        }
    )


    # Check entity_id
    assert "entity_id" in df.columns

    # Check original columns
    assert "business_name" in df.columns
    assert "business_address" in df.columns
    assert "country" in df.columns

    # Check normalized columns
    missing_columns = [
        col
        for col in EXPECTED_NEW_COLUMNS
        if col not in df.columns
    ]

    assert len(missing_columns) == 0, (
        f"Missing columns: {missing_columns}"
    )


    print("✓ entity_id present")
    print("✓ Original columns present")
    print("✓ Normalized columns present")
    print("✓ Rows:", len(df))

    del df
    gc.collect()


print("\nVerification complete.")


----------------------------------------------------------------------
VERIFYING: train_s1
✓ entity_id present
✓ Original columns present
✓ Normalized columns present
✓ Rows: 2206821

----------------------------------------------------------------------
VERIFYING: train_s2
✓ entity_id present
✓ Original columns present
✓ Normalized columns present
✓ Rows: 5034616

----------------------------------------------------------------------
VERIFYING: train_s3
✓ entity_id present
✓ Original columns present
✓ Normalized columns present
✓ Rows: 5285603

----------------------------------------------------------------------
VERIFYING: test_s1
✓ entity_id present
✓ Original columns present
✓ Normalized columns present
✓ Rows: 1732544

----------------------------------------------------------------------
VERIFYING: test_s2
✓ entity_id present
✓ Original columns present
✓ Normalized columns present
✓ Rows: 4887273

----------------------------------------------------------------------
VERIFYING:

In [11]:
print("Cleaned datasets saved in:")
print(CLEANED_DATA_ROOT)

print("\nFiles:")

for filename in sorted(
    os.listdir(CLEANED_DATA_ROOT)
):

    filepath = os.path.join(
        CLEANED_DATA_ROOT,
        filename
    )

    size_mb = (
        os.path.getsize(filepath)
        / (1024 ** 2)
    )

    print(
        f"{filename:<30} "
        f"{size_mb:.2f} MB"
    )

Cleaned datasets saved in:
/content/drive/MyDrive/Amazon ML Challenge 2026/03_Experiments/Cleaned_Data

Files:
test_s1_cleaned.tsv            313.89 MB
test_s2_cleaned.tsv            906.73 MB
test_s3_cleaned.tsv            903.16 MB
train_s1_cleaned.tsv           375.15 MB
train_s2_cleaned.tsv           869.56 MB
train_s3_cleaned.tsv           898.32 MB


In [12]:
CORE_SUFFIXES = {
    "ltd",
    "limited",

    "inc",
    "incorporated",

    "corp",
    "corporation",

    "llc",
    "llp",
    "plc",

    "pvt",
    "private",

    "co",
    "company",

    "org",
    "organization"
}


def create_name_core(name_series):

    result = (
        name_series
        .fillna("")
        .astype("string")
        .str.strip()
    )

    # Remove suffixes only when they occur as complete
    # words at the END of the business name.
    for suffix in CORE_SUFFIXES:

        result = result.str.replace(
            rf"\s+\b{re.escape(suffix)}\b$",
            "",
            regex=True
        )

    # Clean any whitespace left after removal
    result = (
        result
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

    return result

In [13]:
import os
import gc
import pandas as pd


CLEANED_DATA_ROOT = os.path.join(
    DRIVE_ROOT,
    "03_Experiments",
    "Cleaned_Data"
)


CLEANED_FILES = [
    "train_s1_cleaned.tsv",
    "train_s2_cleaned.tsv",
    "train_s3_cleaned.tsv",
    "test_s1_cleaned.tsv",
    "test_s2_cleaned.tsv",
    "test_s3_cleaned.tsv"
]


for filename in CLEANED_FILES:

    print("\n" + "=" * 70)
    print("Processing:", filename)
    print("=" * 70)

    filepath = os.path.join(
        CLEANED_DATA_ROOT,
        filename
    )

    df = pd.read_csv(
        filepath,
        sep="\t",
        dtype={
            "entity_id": "string",
            "business_name": "string",
            "business_address": "string",
            "country": "string",
            "name_norm": "string",
            "address_norm": "string",
            "country_norm": "string"
        }
    )

    original_rows = len(df)

    print("Rows:", original_rows)

    df["name_core"] = create_name_core(
        df["name_norm"]
    )

    assert len(df) == original_rows

    assert "entity_id" in df.columns

    print("name_core added.")
    print("Row count preserved:", len(df))

    df.to_csv(
        filepath,
        sep="\t",
        index=False
    )

    print("Saved:", filepath)

    del df
    gc.collect()

    print("RAM released.")


print("\n" + "=" * 70)
print("name_core added to all cleaned datasets.")
print("=" * 70)


Processing: train_s1_cleaned.tsv
Rows: 2206821
name_core added.
Row count preserved: 2206821
Saved: /content/drive/MyDrive/Amazon ML Challenge 2026/03_Experiments/Cleaned_Data/train_s1_cleaned.tsv
RAM released.

Processing: train_s2_cleaned.tsv
Rows: 5034616
name_core added.
Row count preserved: 5034616
Saved: /content/drive/MyDrive/Amazon ML Challenge 2026/03_Experiments/Cleaned_Data/train_s2_cleaned.tsv
RAM released.

Processing: train_s3_cleaned.tsv
Rows: 5285603
name_core added.
Row count preserved: 5285603
Saved: /content/drive/MyDrive/Amazon ML Challenge 2026/03_Experiments/Cleaned_Data/train_s3_cleaned.tsv
RAM released.

Processing: test_s1_cleaned.tsv
Rows: 1732544
name_core added.
Row count preserved: 1732544
Saved: /content/drive/MyDrive/Amazon ML Challenge 2026/03_Experiments/Cleaned_Data/test_s1_cleaned.tsv
RAM released.

Processing: test_s2_cleaned.tsv
Rows: 4887273
name_core added.
Row count preserved: 4887273
Saved: /content/drive/MyDrive/Amazon ML Challenge 2026/03_Exp

In [14]:
sample_file = os.path.join(
    CLEANED_DATA_ROOT,
    "train_s1_cleaned.tsv"
)

df_check = pd.read_csv(
    sample_file,
    sep="\t",
    dtype={
        "business_name": "string",
        "name_norm": "string",
        "name_core": "string"
    }
)

print(
    df_check[
        [
            "business_name",
            "name_norm",
            "name_core"
        ]
    ].head(30).to_string(index=False)
)

del df_check
gc.collect()

                           business_name                              name_norm                              name_core
                     Orelee's Barbershop                    orelee s barbershop                    orelee s barbershop
                             Prime Money                            prime money                            prime money
                           B+ Retail Inc                           b retail inc                               b retail
                           Christ Chapel                          christ chapel                          christ chapel
                 Prabhav Business Center                prabhav business center                prabhav business center
              Custom Wealth Services LLC             custom wealth services llc                 custom wealth services
Consulting Nyasa Nursing Private Limited       consulting nyasa nursing pvt ltd               consulting nyasa nursing
                       Nexus Anchor Rain        

0